# 01_data_cleaning 
### Premier League Match Dataset (2000/01 – 2024/25)

---

**Notebook Purpose:** This is the first notebook of the project. It loads the raw dataset, checks the data for any issues, creates some new columns I will need later, and saves a clean copy ready for analysis.

**Input file:** `epl_final.csv` with 9,380 Premier League matches  

**Output file:** `epl_cleaned.csv` same matches with additional columns  

*Work done by Manuel Lopez Paz*  😁 *. This is my first data analysis project using Python and Jupyter. I tried to document everything clearly as possible.*

---


In [ ]:
# import librerias
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# config para ver todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('librerías loaded')

librerías loaded


---
## 1. Data Loading

I start by reading the CSV file and looking at its basic structure: how many rows and columns there are, what type pandas assigned to each column, and whether everything loaded correctly.

In [2]:
# cargar el dataset con fechas raw
# despues de controlar el dataset raw, modificare el formato de las fechas

df = pd.read_csv('epl_final.csv')

print(df.shape)
print(f'temporadas en el archivo: {df["Season"].nunique()}')

(9380, 22)
temporadas en el archivo: 25


In [3]:
# verificacion de carga del dataset

df.head()


,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,HomeShots,AwayShots,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,17,8,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,17,12,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,6,16,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,6,13,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,17,12,8,6,6,4,21,20,1,3,0,0


In [4]:
# revisar nombres de columnas y tipos de datos
# hay qie convertir MatchDate aparece como object (texto)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9380 entries, 0 to 9379
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Season             9380 non-null   str  
 1   MatchDate          9380 non-null   str  
 2   HomeTeam           9380 non-null   str  
 3   AwayTeam           9380 non-null   str  
 4   FullTimeHomeGoals  9380 non-null   int64
 5   FullTimeAwayGoals  9380 non-null   int64
 6   FullTimeResult     9380 non-null   str  
 7   HalfTimeHomeGoals  9380 non-null   int64
 8   HalfTimeAwayGoals  9380 non-null   int64
 9   HalfTimeResult     9380 non-null   str  
 10  HomeShots          9380 non-null   int64
 11  AwayShots          9380 non-null   int64
 12  HomeShotsOnTarget  9380 non-null   int64
 13  AwayShotsOnTarget  9380 non-null   int64
 14  HomeCorners        9380 non-null   int64
 15  AwayCorners        9380 non-null   int64
 16  HomeFouls          9380 non-null   int64
 17  AwayFouls          9380 n

### Column Dictionary

The table below defines every column in the raw dataset. 

| # | Column | Type | Description |
|---|--------|------|-------------|
| 1 | `Season` | string | Premier League season in `YYYY/YY` format (e.g. `2023/24`) |
| 2 | `MatchDate` | string -> datetime | Calendar date on which the match was played |
| 3 | `HomeTeam` | string | Name of the team playing at their home ground |
| 4 | `AwayTeam` | string | Name of the visiting team |
| 5 | `FullTimeHomeGoals` | integer | Goals scored by the home team at full time |
| 6 | `FullTimeAwayGoals` | integer | Goals scored by the away team at full time |
| 7 | `FullTimeResult` | string | Full-time result: **H** = home win, **D** = draw, **A** = away win |
| 8 | `HalfTimeHomeGoals` | integer | Home goals scored in the first half |
| 9 | `HalfTimeAwayGoals` | integer | Away goals scored in the first half |
| 10 | `HalfTimeResult` | string | Half-time result: **H**, **D**, or **A** |
| 11 | `HomeShots` | integer | Total shots attempted by the home team (on and off target) |
| 12 | `AwayShots` | integer | Total shots attempted by the away team |
| 13 | `HomeShotsOnTarget` | integer | Home shots that required a save or resulted in a goal |
| 14 | `AwayShotsOnTarget` | integer | Away shots on target |
| 15 | `HomeCorners` | integer | Corner kicks awarded to the home team |
| 16 | `AwayCorners` | integer | Corner kicks awarded to the away team |
| 17 | `HomeFouls` | integer | Fouls committed by the home team |
| 18 | `AwayFouls` | integer | Fouls committed by the away team |
| 19 | `HomeYellowCards` | integer | Yellow cards received by home team players |
| 20 | `AwayYellowCards` | integer | Yellow cards received by away team players |
| 21 | `HomeRedCards` | integer | Red cards received by home team players |
| 22 | `AwayRedCards` | integer | Red cards received by away team players |

> **Note on shots:** To clarify `ShotsOnTarget` is a subset of `Shots`. This means every shot on target is also a shot, but not every shot is on target. Shot accuracy (obtained in Section 3) measures the proportion that were on target.

---
## 2. Data Quality Check

Before changing anything in the data, I want to check whether the raw file has any obvious problems: missing values, duplicate rows, dates outside the expected range, or result codes that are not H, D, or A.

### 2.1 Missing Value Check

In [5]:
# verificar valores faltantes en el dataset
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

tabla_nulos = pd.DataFrame({'faltantes': nulos, 'porcentaje': pct_nulos})
print(tabla_nulos)
print()

if nulos.sum() == 0:
    print('sin valores faltantes')
else:
    print(f'{nulos.sum()} valores faltantes encontrados')

                   faltantes  porcentaje
Season                     0      0.0000
MatchDate                  0      0.0000
HomeTeam                   0      0.0000
AwayTeam                   0      0.0000
FullTimeHomeGoals          0      0.0000
FullTimeAwayGoals          0      0.0000
FullTimeResult             0      0.0000
HalfTimeHomeGoals          0      0.0000
HalfTimeAwayGoals          0      0.0000
HalfTimeResult             0      0.0000
HomeShots                  0      0.0000
AwayShots                  0      0.0000
HomeShotsOnTarget          0      0.0000
AwayShotsOnTarget          0      0.0000
HomeCorners                0      0.0000
AwayCorners                0      0.0000
HomeFouls                  0      0.0000
AwayFouls                  0      0.0000
HomeYellowCards            0      0.0000
AwayYellowCards            0      0.0000
HomeRedCards               0      0.0000
AwayRedCards               0      0.0000

sin valores faltantes


### 2.2 Duplicate Row Check

In [6]:
# verificar filas duplicadas, exactas y por (fecha + equipos)
dupes_exactos = df.duplicated().sum()
dupes_logicos = df.duplicated(subset=['MatchDate', 'HomeTeam', 'AwayTeam']).sum()

print(f'filas exactamente duplicadas: {dupes_exactos}')
print(f'partidos duplicados (misma fecha + equipos): {dupes_logicos}')
print()

if dupes_exactos == 0 and dupes_logicos == 0:
    print('sin duplicados')
else:
    print('REVISAR; hay duplicados')

filas exactamente duplicadas: 0
partidos duplicados (misma fecha + equipos): 0

sin duplicados


### 2.3 Date Range and Season Coverage

In [7]:
# rango de fechas y cobertura por temporada
fechas = pd.to_datetime(df['MatchDate'])

print(f'fecha mas vieja: {fechas.min().date()}')
print(f'fecha mas reciente: {fechas.max().date()}')
print()

# contar partidos por temporada para detectar temporadas incompletas
partidos_por_temporada = df.groupby('Season').size().reset_index()
partidos_por_temporada.columns = ['temporada', 'partidos']

# ordenar cronologicamente
partidos_por_temporada['anio'] = partidos_por_temporada['temporada'].apply(
    lambda s: int(s.split('/')[0])
)
partidos_por_temporada = (partidos_por_temporada
    .sort_values('anio')
    .drop(columns='anio')
    .reset_index(drop=True)
)

print(f'numero de temporadas: {len(partidos_por_temporada)}')
print()
print(partidos_por_temporada.to_string(index=False))
print()
print(f'total de partidos: {partidos_por_temporada["partidos"].sum()}')

fecha mas vieja: 2000-08-19
fecha mas reciente: 2025-05-05

numero de temporadas: 25

temporada  partidos
  2000/01       380
  2001/02       380
  2002/03       380
  2003/04       335
  2004/05       335
  2005/06       380
  2006/07       380
  2007/08       380
  2008/09       380
  2009/10       380
  2010/11       380
  2011/12       380
  2012/13       380
  2013/14       380
  2014/15       380
  2015/16       380
  2016/17       380
  2017/18       380
  2018/19       380
  2019/20       380
  2020/21       380
  2021/22       380
  2022/23       380
  2023/24       380
  2024/25       350

total de partidos: 9380


**2003/04 and 2004/05** these seasons show fewer than 380 matches in the dataset. The Premier League played the full 380 matches both seasons, so the lower count means the source file is missing some records for those years.

### 2.4 Category Check - Results

In [8]:
# verificar que los codigos de resultado sean solo H, D o A
valores_validos = {'H', 'D', 'A'}

for col in ['FullTimeResult', 'HalfTimeResult']:
    valores = set(df[col].unique())
    invalidos = valores - valores_validos
    conteo = df[col].value_counts().to_dict()

    print(f'{col}:')
    print(f'  valores únicos: {sorted(valores)}')
    print(f'  conteo: {conteo}')

    if invalidos:
        print(f'  error — valores inesperados: {invalidos}')
    else:
        print(f'  ok — solo H / D / A')
    print()

FullTimeResult:
  valores únicos: ['A', 'D', 'H']
  conteo: {'H': 4299, 'A': 2768, 'D': 2313}
  ok — solo H / D / A

HalfTimeResult:
  valores únicos: ['A', 'D', 'H']
  conteo: {'D': 3840, 'H': 3290, 'A': 2250}
  ok — solo H / D / A



### 2.5 Numeric Range Check

In [9]:
# verificar rangos logicos en columnas numericas
cols_numericas = df.select_dtypes(include='number').columns.tolist()

# (a) buscar valores negativos
negativos = {col: (df[col] < 0).sum() for col in cols_numericas}
neg_encontrados = {k: v for k, v in negativos.items() if v > 0}

print('valores negativos:')
if neg_encontrados:
    print(f'  encontrados: {neg_encontrados}')
else:
    print('  ninguno — ok')
print()

# (b) revisar si ShotsOnTarget supera Shots — imposible
filas_home = df[df['HomeShotsOnTarget'] > df['HomeShots']]
filas_away = df[df['AwayShotsOnTarget'] > df['AwayShots']]

print(f'partidos donde HomeShotsOnTarget > HomeShots: {len(filas_home)}')
print(f'partidos donde AwayShotsOnTarget > AwayShots: {len(filas_away)}')
print()

if len(filas_home) > 0:
    print('filas problematicas (local):')
    display(filas_home[['Season', 'MatchDate', 'HomeTeam', 'AwayTeam',
                         'HomeShots', 'HomeShotsOnTarget']].reset_index(drop=True))
    print()

if len(filas_away) > 0:
    print('filas problematicas (visitante):')
    display(filas_away[['Season', 'MatchDate', 'HomeTeam', 'AwayTeam',
                         'AwayShots', 'AwayShotsOnTarget']].reset_index(drop=True))
    print()

print('parece un error de ingreso de datos, correjido en la seccion 3.5')
print()

# (c) partidos con cero tiros - NaN en precision de tiro
tiros_cero_home = (df['HomeShots'] == 0).sum()
tiros_cero_away = (df['AwayShots'] == 0).sum()

print(f'partidos con HomeShots == 0: {tiros_cero_home}')
print(f'partidos con AwayShots == 0: {tiros_cero_away}')
print('precision de tiro sera NaN en esas filas (ver sección 3.5)')

valores negativos:
  ninguno — ok

partidos donde HomeShotsOnTarget > HomeShots: 2
partidos donde AwayShotsOnTarget > AwayShots: 3

filas problematicas (local):


,Season,MatchDate,HomeTeam,AwayTeam,HomeShots,HomeShotsOnTarget
0,2000/01,2000-10-14,Coventry,Tottenham,0,5
1,2000/01,2001-04-13,Bradford,Charlton,4,8



filas problematicas (visitante):


,Season,MatchDate,HomeTeam,AwayTeam,AwayShots,AwayShotsOnTarget
0,2000/01,2000-10-14,Coventry,Tottenham,4,5
1,2000/01,2001-04-13,Bradford,Charlton,3,8
2,2021/22,2021-08-15,Newcastle,West Ham,8,9



parece un error de ingreso de datos, correjido en la seccion 3.5

partidos con HomeShots == 0: 2
partidos con AwayShots == 0: 6
precision de tiro sera NaN en esas filas (ver sección 3.5)


### 2.6 Summary Statistics

In [10]:
# estadisticas descriptivas de las columnas numericas
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
FullTimeHomeGoals,9380.0000,1.5350,1.3050,0.0000,1.0000,1.0000,2.0000,9.0000
FullTimeAwayGoals,9380.0000,1.1830,1.1570,0.0000,0.0000,1.0000,2.0000,9.0000
HalfTimeHomeGoals,9380.0000,0.6880,0.8350,0.0000,0.0000,0.0000,1.0000,5.0000
HalfTimeAwayGoals,9380.0000,0.5190,0.7350,0.0000,0.0000,0.0000,1.0000,5.0000
HomeShots,9380.0000,13.6170,5.3560,0.0000,10.0000,13.0000,17.0000,43.0000
AwayShots,9380.0000,10.8110,4.6970,0.0000,7.0000,10.0000,14.0000,37.0000
HomeShotsOnTarget,9380.0000,5.9730,3.2680,0.0000,4.0000,6.0000,8.0000,24.0000
AwayShotsOnTarget,9380.0000,4.6940,2.7500,0.0000,3.0000,4.0000,6.0000,20.0000
HomeCorners,9380.0000,6.0400,3.1110,0.0000,4.0000,6.0000,8.0000,20.0000
AwayCorners,9380.0000,4.7750,2.7500,0.0000,3.0000,4.0000,6.0000,19.0000


*The check looks good. No nulls, no duplicates, and the result codes are all correct. I was surprised there are zero missing values. The only issue is those five rows where shots on target exceed total shots, which I will fix in the next section.*

**Reading the summary table:**

- **Goals:** Home teams average 1.54 goals per game versus 1.18 for away teams. That gap of 0.36 goals is the home advantage showing up in the data.
- **Shots:** Home teams take 13.6 shots per game on average versus 10.8 for away teams, so home sides are consistently more active in attack.
- **Shots on Target:** Home teams: 5.97 per game. Away teams: 4.69 per game. I will use these in the regression model in `02_analysis_EPL.ipynb`.
- **Red Cards:** Very rare. The average is less than 0.1 per game (so I am not going to use them as predictors).
- **Fouls:** Both teams commit roughly 11–12 fouls per game, so there is no big difference there.

---
## 3. Data Cleaning and Adding Variables

The raw data passed all the quality checks, so I only need to make two types of changes:

1. **Fixing a data type**: `MatchDate` is stored as string, but I need it as a proper date so I can filter and group by date.
2. **Creating new columns**: I will build new variables from the existing ones because they are more useful for the analysis than the raw columns on their own.

Worked on a copy of the dataframe so the original stays untouched.

### 3.1 Parse Match Date (string to datetime)

In [11]:
# MatchDate de texto a datetime 
df_clean = df.copy()

df_clean['MatchDate'] = pd.to_datetime(df_clean['MatchDate'])

print(f'tipo antes: {df["MatchDate"].dtype}')
print(f'tipo despues: {df_clean["MatchDate"].dtype}')
print()
print('muestra de fechas convertidas:')
print(df_clean['MatchDate'].head(3))

tipo antes: str
tipo despues: datetime64[us]

muestra de fechas convertidas:
0   2000-08-19
1   2000-08-19
2   2000-08-19
Name: MatchDate, dtype: datetime64[us]


### 3.2 COVID Period Flag

The Premier League was suspended in March 2020 because of COVID-19. When it restarted in June 2020 and throughout the whole 2020/21 season, all matches were played without any supporters inside the stadiums.

I want to mark exactly those matches with a flag so I can compare them to normal matches later. Rather than flagging entire seasons, I use the actual match dates for accuracy. That way the 2019/20 matches played normally in August–February (with fans inside the stadiums) are not accidentally labelled as COVID matches.

**COVID window:** `2020-06-17` → `2021-05-17`  
- **Start:** The date of the first fixture played without fans  
- **End:** The last day before the general reopening of stadiums. From May 18, 2021, limited attendance (up to 10,000 spectators) was allowed, and the final two matchdays of the 2020/21 season were played with home fans in the stadiums.

In [ ]:
# crear indicador COVID basado en fechas
# marca los partidos jugados sin aficionados entre junio 2020 y mayo 2021

COVID_START = pd.Timestamp('2020-06-17')
COVID_END   = pd.Timestamp('2021-05-17')

# comparacion de fechas 
en_periodo_covid = (df_clean['MatchDate'] >= COVID_START) & (df_clean['MatchDate'] <= COVID_END)
df_clean['COVID'] = en_periodo_covid.astype(int)

# verificar cuantos partidos quedaron marcados
n_covid  = df_clean['COVID'].sum()
n_normal = len(df_clean) - n_covid

print(f'partidos sin aficionados (COVID=1): {n_covid}')
print(f'partidos normales (COVID=0): {n_normal}')
print()

# desglose por temporada
desglose = (
    df_clean[df_clean['COVID'] == 1]
    .groupby('Season')['MatchDate']
    .agg(primero='min', ultimo='max', partidos='count')
    .reset_index()
)
print(desglose.to_string(index=False))

partidos sin aficionados (COVID=1): 452
partidos normales (COVID=0): 8928

 Season    primero     ultimo  partidos
2019/20 2020-06-17 2020-07-26        92
2020/21 2020-09-12 2021-05-16       360


### 3.3 Goal Based Variables

In [13]:
# diferencia de goles: positivo = gana local, 0 = empate, negativo = gana visitante
df_clean['GoalDiff'] = df_clean['FullTimeHomeGoals'] - df_clean['FullTimeAwayGoals']

# total de goles del partido
df_clean['TotalGoals'] = df_clean['FullTimeHomeGoals'] + df_clean['FullTimeAwayGoals']

print(f'GoalDiff  — min: {df_clean["GoalDiff"].min()}, max: {df_clean["GoalDiff"].max()}, media: {df_clean["GoalDiff"].mean():.4f}')
print(f'TotalGoals — min: {df_clean["TotalGoals"].min()}, max: {df_clean["TotalGoals"].max()}, media: {df_clean["TotalGoals"].mean():.4f}')
print()

# comprobacion GoalDiff debe coincidir con FullTimeResult
ok = (
    ((df_clean['GoalDiff'] > 0) == (df_clean['FullTimeResult'] == 'H')) &
    ((df_clean['GoalDiff'] == 0) == (df_clean['FullTimeResult'] == 'D')) &
    ((df_clean['GoalDiff'] < 0) == (df_clean['FullTimeResult'] == 'A'))
).all()
print(f'consistencia con FullTimeResult: {"ok" if ok else "revisar"}')

GoalDiff  — min: -9, max: 9, media: 0.3528
TotalGoals — min: 0, max: 11, media: 2.7180

consistencia con FullTimeResult: ok


### 3.4 Result Encoding Variables

In [ ]:
# indicador binario de victoria local
df_clean['HomeWin'] = (df_clean['FullTimeResult'] == 'H').astype(int)

# codificacion numerica del resultado: victoria local=1, empate=0, victoria visitante=-1
mapa_resultado = {'H': 1, 'D': 0, 'A': -1}
df_clean['ResultNumeric'] = df_clean['FullTimeResult'].map(mapa_resultado)

print('columnas creadas: HomeWin, ResultNumeric')
print()

# verificacion de la codificacion
tabla_check = (df_clean[['FullTimeResult', 'HomeWin', 'ResultNumeric']]
    .drop_duplicates()
    .sort_values('ResultNumeric', ascending=False))
print(tabla_check.to_string(index=False))
print()
print(f'partidos ganados por el local: {df_clean["HomeWin"].sum()} ({df_clean["HomeWin"].mean():.1%})')

columnas creadas: HomeWin, ResultNumeric

FullTimeResult  HomeWin  ResultNumeric
             H        1              1
             D        0              0
             A        0             -1

partidos ganados por el local: 4299 (45.8%)


### 3.5 Shot Accuracy

In [ ]:
# precision de tiro. Hay algunos partidos con Shots == 0
# en esos casos dejo NaN en vez de dividir por cero

# (a) limitar ShotsOnTarget para que no supere Shots
sot_home = df_clean['HomeShotsOnTarget'].clip(upper=df_clean['HomeShots'])
sot_away = df_clean['AwayShotsOnTarget'].clip(upper=df_clean['AwayShots'])

# (b) calcular precision, NaN donde Shots == 0
df_clean['HomeShotAccuracy'] = np.where(
    df_clean['HomeShots'] > 0,
    sot_home / df_clean['HomeShots'],
    np.nan
)

df_clean['AwayShotAccuracy'] = np.where(
    df_clean['AwayShots'] > 0,
    sot_away / df_clean['AwayShots'],
    np.nan
)

# verificacion
print(f'HomeShotAccuracy — media: {df_clean["HomeShotAccuracy"].mean():.4f}, NaN: {df_clean["HomeShotAccuracy"].isna().sum()}')
print(f'AwayShotAccuracy — media: {df_clean["AwayShotAccuracy"].mean():.4f}, NaN: {df_clean["AwayShotAccuracy"].isna().sum()}')

# confirmar que ningun valor suppera 1.0
max_acc = max(df_clean['HomeShotAccuracy'].max(), df_clean['AwayShotAccuracy'].max())
print(f'valor maximo de precision: {max_acc:.4f}  (tiene que ser ≤ 1.0)')

HomeShotAccuracy — media: 0.4416, NaN: 2
AwayShotAccuracy — media: 0.4398, NaN: 6
valor máximo de precisión: 1.0000  (debe ser ≤ 1.0)


### New Variables Summary

Here is a quick reference for every new column I created and why.

| New Column | Formula | Type | Purpose |
|---|---|---|---|
| `COVID` | `1 if 2020-06-17 ≤ MatchDate ≤ 2021-05-17 else 0` | Binary int | Marks the 452 matches played without fans |
| `GoalDiff` | `FullTimeHomeGoals − FullTimeAwayGoals` | Signed int | Used as the dependent variable in the regression |
| `TotalGoals` | `FullTimeHomeGoals + FullTimeAwayGoals` | Positive int | Total goals in the match; used in the time-trend charts |
| `HomeWin` | `1 if FullTimeResult == 'H' else 0` | Binary int | Used in the t-tests to compare home win rates |
| `ResultNumeric` | `H→1, D→0, A→−1` | Ordinal int | Lets me calculate correlations with the result |
| `HomeShotAccuracy` | `HomeShotsOnTarget / HomeShots` (NaN if 0 shots) | Float [0,1] | How accurate the home team's shooting was |
| `AwayShotAccuracy` | `AwayShotsOnTarget / AwayShots` (NaN if 0 shots) | Float [0,1] | How accurate the away team's shooting was |

> **Why date-based and not season-based for COVID?** If I had flagged the whole 2019/20 and 2020/21 seasons, I would have incorrectly included 288 matches from 2019/20 that were played normally in front of fans before the March 2020 suspension. Using dates gives me 452 truly fan-free matches instead of 760 mixed ones.

---
## 4. Cleaned Dataset Overview

I finish this notebook by checking the cleaned dataframe and saving it. The exported file (`epl_cleaned.csv`) is the single source of for all subsequent analysis in this project.

In [16]:
# dimensiones finales del dataset limpio
print(f'original: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'limpio:   {df_clean.shape[0]} filas × {df_clean.shape[1]} columnas')
print(f'columnas nuevas añadidas: {df_clean.shape[1] - df.shape[1]}')
print()

# lista completa de columnas con su origen
print('columnas del dataset limpio:')
for i, col in enumerate(df_clean.columns, 1):
    marca = '* nueva' if col not in df.columns else '       '
    print(f'  {marca}  {i:2d}. {col}')

original: 9380 filas × 22 columnas
limpio:   9380 filas × 29 columnas
columnas nuevas añadidas: 7

columnas del dataset limpio:
            1. Season
            2. MatchDate
            3. HomeTeam
            4. AwayTeam
            5. FullTimeHomeGoals
            6. FullTimeAwayGoals
            7. FullTimeResult
            8. HalfTimeHomeGoals
            9. HalfTimeAwayGoals
           10. HalfTimeResult
           11. HomeShots
           12. AwayShots
           13. HomeShotsOnTarget
           14. AwayShotsOnTarget
           15. HomeCorners
           16. AwayCorners
           17. HomeFouls
           18. AwayFouls
           19. HomeYellowCards
           20. AwayYellowCards
           21. HomeRedCards
           22. AwayRedCards
  * nueva  23. COVID
  * nueva  24. GoalDiff
  * nueva  25. TotalGoals
  * nueva  26. HomeWin
  * nueva  27. ResultNumeric
  * nueva  28. HomeShotAccuracy
  * nueva  29. AwayShotAccuracy


In [17]:
# ver solo las columnas nuevas para verificar

preview_cols = [
    'Season', 'MatchDate', 'FullTimeResult',
    'COVID', 'GoalDiff', 'TotalGoals',
    'HomeWin', 'ResultNumeric',
    'HomeShotAccuracy', 'AwayShotAccuracy'
]

print('Preview — new and modified columns (first 8 rows):\n')
display(df_clean[preview_cols].head(8))

Preview — new and modified columns (first 8 rows):



,Season,MatchDate,FullTimeResult,COVID,GoalDiff,TotalGoals,HomeWin,ResultNumeric,HomeShotAccuracy,AwayShotAccuracy
0,2000/01,2000-08-19,H,0,4,4,1,1,0.8235,0.5000
1,2000/01,2000-08-19,H,0,2,6,1,1,0.5882,0.4167
2,2000/01,2000-08-19,A,0,-2,4,0,-1,0.5000,0.5625
3,2000/01,2000-08-19,D,0,0,4,0,0,0.6667,0.4615
4,2000/01,2000-08-19,H,0,2,2,1,1,0.4706,0.5000
5,2000/01,2000-08-19,D,0,0,0,0,0,0.8000,0.6000
6,2000/01,2000-08-19,H,0,1,1,1,1,0.6250,0.6667
7,2000/01,2000-08-19,H,0,1,1,1,1,0.2500,0.5000


In [ ]:
# verificar nulos en el dataset final
# solo deben aparecer en las columnas de precision de tiro
nulos_finales = df_clean.isnull().sum()
nulos_finales = nulos_finales[nulos_finales > 0]

if nulos_finales.empty:
    print('sin valores faltantes en el dataset limpio')
else:
    print('columnas con NaN en el dataset limpio:')
    for col, n in nulos_finales.items():
        print(f'  {col}: {n} NaN ({n/len(df_clean)*100:.3f}%)')
    print()
    print('esperado: son partidos sin tiros registrados (ver sección 3.5)')

columnas con NaN en el dataset limpio:
  HomeShotAccuracy: 2 NaN (0.021%)
  AwayShotAccuracy: 6 NaN (0.064%)

esperado: son partidos sin tiros registrados (ver sección 3.5)


In [ ]:
# exportar el dataset limpio a CSV
# index=False evita agregar una columna de indice innecesaria

ruta_limpio = 'epl_cleaned.csv'
df_clean.to_csv(ruta_limpio, index=False)

print(f'archivo guardado: {ruta_limpio}')

# verificacion rapida de que se guardo bien
df_check = pd.read_csv(ruta_limpio)
print(f'verificacion: {df_check.shape[0]} filas × {df_check.shape[1]} columnas')

archivo guardado: epl_cleaned.csv
verificación: 9380 filas × 29 columnas


---
## Notebook Summary

### What I did

| Stage | Action | Result |
|---|---|---|
| **Data Loading** | Loaded CSV, checked column names and types | 9,380 rows × 22 columns confirmed |
| **Quality Check** | Checked for nulls, duplicates, bad codes, impossible values | All clear; 5 shot inconsistency rows and 8 zero-shot rows noted |
| **Date Parsing** | Converted `MatchDate` from text → `datetime64` | Needed so I can filter by date and build time charts |
| **COVID Flag** | Date-based binary column: `2020-06-17` → `2021-05-17` | 452 COVID matches; 8,928 normal matches |
| **Goal Variables** | Made `GoalDiff` and `TotalGoals` | Cross-checked against `FullTimeResult` |
| **Result Encoding** | Made `HomeWin` (0/1) and `ResultNumeric` (−1/0/1) | 45.8 % home win rate overall |
| **Shot Accuracy** | Divided shots on target by total shots | NaN where shots = 0 (capped at 1.0 for the 5 bad rows) |
| **Export** | Saved `epl_cleaned.csv` and reloaded to check | 9,380 rows × 29 columns |

### Why I made these choices

- **No rows were deleted.** All 9,380 matches stayed in. The zero-shot rows are kept in the dataset. A match with no shots recorded is extreme but not impossible. The five rows where shots on target exceed total shots are a data error. Rather than deleting those rows I set shots on target equal to total shots (only for this five rows).

- **Date-based COVID flag.** Using season labels would have marked the first 288 matches of 2019/20 as COVID matches when they were actually played with fans. The date range gives me only the 452 matches that were genuinely without fans.

- **NaN instead of 0 for undefined shot accuracy.** If I had put 0 for matches where no shots were recorded, it would drag down the average. NaN tells pandas to skip those rows when calculating means.

- **Capping shot accuracy at 1.0.** Five rows have more shots on target recorded than total shots, which cannot be right. Rather than deleting those rows I set shots on target equal to total shots for those five cases and keep the rows in the dataset.

- **GoalDiff as the main outcome.** Using a continuous number instead of three categories simplifies the regression model. I discuss the limitation of using integers (whole numbers) in `02_analysis.ipynb`.

---
*Work continues on `02_analysis.ipynb` where I do the charts and statistical tests.*